# NirmaanAI — EDA 01: AI4I 2020 Predictive Maintenance
**Module**: Phase 6 (Predictive Maintenance) & Phase 11/12 (XAI/Root Cause Analysis)
**Dataset**: `DATASET/01_AI4I_2020/raw/ai4i2020.csv`
**Institution**: IEM Kolkata, CSE (AI), Group 59
**Guide**: PROF. KUNTAL MONDAL


In [ ]:
import os, sys
sys.path.append(r'C:/NIRMAAN AI')
import pandas as pd
import numpy as np
from src.data.profiling import profile_dataframe, compute_correlation_matrix

csv_path = r'C:/NIRMAAN AI/DATASET/01_AI4I_2020/raw/ai4i2020.csv'
df = pd.read_csv(csv_path)
print(f'AI4I 2020 Shape: {df.shape}')
df.head()


## 1. Statistical Profile & Target Class Imbalance
Examine class imbalance between normal operations and machine failures.


In [ ]:
profile = profile_dataframe(df, target_col='Machine failure')
print(f'Rows: {profile["rows"]}, Cols: {profile["cols"]}')
print('Target Distribution:', profile['target_summary']['distribution'])
print(f'Imbalance Ratio: {profile["target_summary"]["imbalance_ratio"]}:1')


## 2. Failure Mode Analysis & Target Leakage Prevention
Failure modes (TWF, HDF, PWF, OSF, RNF) are root causes, NOT input features!


In [ ]:
failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
mode_counts = {m: int(df[m].sum()) for m in failure_modes}
print('Failure Mode Frequencies:', mode_counts)

# Calculate operational temperature difference and power
df['Temp_Diff_K'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power_kW'] = (2 * np.pi * df['Rotational speed [rpm]'] * df['Torque [Nm]']) / 60000.0
print('Physical Feature Correlations with Failure:')
print(df[['Machine failure', 'Torque [Nm]', 'Tool wear [min]', 'Power_kW', 'Temp_Diff_K']].corr()['Machine failure'])

# Verified statistics
twf_wear = df[df['TWF'] == 1]['Tool wear [min]']
print(f'Tool wear when TWF=1: min={twf_wear.min()}, mean={twf_wear.mean():.2f}, max={twf_wear.max()}')
hdf_temp = df[df['HDF'] == 1]['Temp_Diff_K']
print(f'Temp_Diff_K when HDF=1: min={hdf_temp.min():.2f}, max={hdf_temp.max():.2f}')


## 3. Key Findings for NirmaanAI Engine
1. **Severe Imbalance**: 3.39% failure rate (28.5:1 ratio) requires PR-AUC and stratified validation.
2. **Tool Wear (TWF)**: 97.8% of TWF failures occur at Tool wear >= 200 minutes.
3. **Heat Dissipation (HDF)**: 100% of HDF failures occur when Temp_Diff_K <= 8.6 K.
